<a href="https://colab.research.google.com/github/jeevan841/Innomatics-Internship/blob/main/NLP_03_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installing all the dependencies
!pip install nltk scikit-learn pandas

import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [10]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
#uploading the file
from google.colab import files
uploaded = files.upload()

Saving imdb_top_1000.csv to imdb_top_1000.csv


In [4]:
#loading and testing the data
df = pd.read_csv(list(uploaded.keys())[0])
df.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [6]:
#data organization
print("Shape:", df.shape)
print("\nColumns:", df.columns)
# Adjust column names if needed
df = df[['Overview', 'IMDB_Rating']]
df.columns = ['text', 'sentiment']
print("\nClass Distribution:\n", df['sentiment'].value_counts())
print("\nSample Text:\n", df['text'][0])

Shape: (1000, 16)

Columns: Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='object')

Class Distribution:
 sentiment
7.7    157
7.8    151
8.0    141
8.1    127
7.6    123
7.9    106
8.2     67
8.3     44
8.4     31
8.5     20
8.6     15
8.7      5
8.8      5
8.9      3
9.0      3
9.3      1
9.2      1
Name: count, dtype: int64

Sample Text:
 Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.


In [7]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return " ".join(tokens)

In [11]:
#preprocessing
df['cleaned_text'] = df['text'].apply(clean_text)
df[['text', 'cleaned_text']].head()

,text,cleaned_text
0,Two imprisoned men bond over a number of years...,two imprisoned men bond number year finding so...
1,An organized crime dynasty's aging patriarch t...,organized crime dynasty aging patriarch transf...
2,When the menace known as the Joker wreaks havo...,menace known joker wreaks havoc chaos people g...
3,The early life and career of Vito Corleone in ...,early life career vito corleone new york city ...
4,A jury holdout attempts to prevent a miscarria...,jury holdout attempt prevent miscarriage justi...


In [12]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_text'])

In [13]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_text'])

In [16]:
def categorize_imdb_rating(rating):
    if rating >= 8.0:
        return 'high_rating'
    else:
        return 'average_rating'

y = df['sentiment'].apply(categorize_imdb_rating)

X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [17]:
#training
# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_tfidf, y_train)
y_pred_dt = dt.predict(X_test_tfidf)

In [18]:
def evaluate(y_test, y_pred, name):
    print(f"🔹 {name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("-" * 40)

In [19]:
evaluate(y_test, y_pred_lr, "Logistic Regression (TF-IDF)")
evaluate(y_test, y_pred_nb, "Naive Bayes (BoW)")
evaluate(y_test, y_pred_dt, "Decision Tree (TF-IDF)")

🔹 Logistic Regression (TF-IDF)
Accuracy: 0.56
Precision: 0.5535533079091361
Recall: 0.56
F1 Score: 0.5108639705882353
----------------------------------------
🔹 Naive Bayes (BoW)
Accuracy: 0.535
Precision: 0.5337581168831168
Recall: 0.535
F1 Score: 0.5342594605525923
----------------------------------------
🔹 Decision Tree (TF-IDF)
Accuracy: 0.47
Precision: 0.46801037534330175
Recall: 0.47
F1 Score: 0.4688197185949995
----------------------------------------


In [20]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_nb, average='weighted'),
        f1_score(y_test, y_pred_dt, average='weighted')
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.560,0.510864
1,Naive Bayes,0.535,0.534259
2,Decision Tree,0.470,0.468820


In [21]:
def predict_sentiment(text):
    cleaned = clean_text(text)
    vector = tfidf.transform([cleaned])
    prediction = lr.predict(vector)
    return prediction[0]

# Try your own sentence
predict_sentiment("This movie was absolutely amazing!")

'average_rating'